In [ ]:
import pandas as pd
import matplotlib.pyplot as plt 
import os
import numpy as np
from sklearn.manifold import TSNE
import re
%matplotlib inline

## Configuration

In [ ]:
dataset_name = "MNIST" # DTD | EuroSAT | GTSRB | MNIST | RESISC45 | Stanford_Cars | SUN397 | SVHN
domain = "Base_Fine_Tuned" # Base_Fine_Tuned | Fine_Tuned_Layer_Skipping
model_name = "CLIP_ViT_Vision" # DeiT | CLIP_ViT_Vision | Google_ViT
model_type = "Fine_Tuned" # Fine_Tuned | Base
layer_norm = "Pre_Post_Layer_Norm" # Pre_Post_Layer_Norm | Post_Layer_Norm
num_classes = 10
num_img = 100

results_path = f"../Data/Embedding_Captures/{dataset_name}/{domain}/{layer_norm}/Entire_Transformation_Matrix_W"
indices = [i for i in range(12)]

## File Prepping

In [ ]:
def extract_class_index(filename):
    match = re.search(r'Standard_1000_Base_(\d+)_Results\.json', filename)
    return int(match.group(1)) if match else -1

In [ ]:
model_type = "Fine_Tuned"
results_fine_tuned = [] # File Loading

try:
    for filename in os.listdir(f"{results_path}/{model_type}"):
        if filename == ".DS_Store":
            continue
        else:
            file_path = os.path.join(f"{results_path}/{model_type}", filename)
            if os.path.isfile(file_path):
                results_fine_tuned.append(file_path)
except FileNotFoundError:
    print(f"Error: The Folder '{results_path}' was not found.")
except Exception as e:
    print(f"An error occured: {e}")

results_fine_tuned = sorted(results_fine_tuned, key=lambda f: extract_class_index(os.path.basename(f)))
results_fine_tuned = [pd.read_json(i) for i in results_fine_tuned]

In [ ]:
model_type = "Base"
results_base = [] # File Loading

try:
    for filename in os.listdir(f"{results_path}/{model_type}"):
        if filename == ".DS_Store":
            continue
        else:
            file_path = os.path.join(f"{results_path}/{model_type}", filename)
            if os.path.isfile(file_path):
                results_base.append(file_path)
except FileNotFoundError:
    print(f"Error: The Folder '{results_path}' was not found.")
except Exception as e:
    print(f"An error occured: {e}")

results_base = sorted(results_base, key=lambda f: extract_class_index(os.path.basename(f)))
results_base = [pd.read_json(i) for i in results_base]

In [ ]:
base_embeddings = {i: {j: [] for j in indices} for i in range(num_classes)}
fine_tuned_embeddings = {i: {j: [] for j in indices} for i in range(num_classes)}

for i in range(num_classes):
    for j in indices:
        for k in range(num_img):
            base_embeddings[i][j].append(np.array(results_base[i].iloc[j]["Embeddings"][k]))
            fine_tuned_embeddings[i][j].append(np.array(results_fine_tuned[i].iloc[j]["Embeddings"][k]))

In [ ]:
task_matrix = []
task_matrix_path = "../Data/Class_Specific_B_F/MNIST/Base_Fine_Tuned/Entire_Transformation_Matrix_W"

try:
    for filename in os.listdir(task_matrix_path):
        if filename in ["Standard_48000_Results_All_Classes.json"]:
            file_path = os.path.join(task_matrix_path, filename)
            if os.path.isfile(file_path):
                task_matrix.append(file_path)
except FileNotFoundError:
    print(f"Error: The Folder '{task_matrix_path}' was not found.")
except Exception as e:
    print(f"An error occured: {e}")

task_matrix = np.array([pd.read_json(i) for i in task_matrix][0]["W"][11])

## Augmentations

In [ ]:
# task_matrix (768, 768)
# fine_tuned_embeddings (10, 12, 100, 768) 
# base_embeddings (10, 12, 100, 768)

In [ ]:
augmented_embeddings = {i: {j: [] for j in indices} for i in range(num_classes)}
for i in range(num_classes):
    for j in indices:
        for k in range(num_img):
            augmented_embeddings[i][j].append(base_embeddings[i][j][k] @ task_matrix)

In [ ]:
for i in range(num_classes):
    for j in indices:
        base_embeddings[i][j] = np.array(base_embeddings[i][j])
        fine_tuned_embeddings[i][j] = np.array(fine_tuned_embeddings[i][j])
        augmented_embeddings[i][j] = np.array(augmented_embeddings[i][j])

## Graphs

In [ ]:
tsne = TSNE(n_components=2, perplexity=5, random_state=42)

for i in range(num_classes):
    base = tsne.fit_transform(base_embeddings[i][11])
    fine_tuned = tsne.fit_transform(fine_tuned_embeddings[i][11])
    augmented = tsne.fit_transform(augmented_embeddings[i][11])

    plt.scatter(base[:,0], base[:,1], color="blue", label="base")
    plt.scatter(fine_tuned[:,0], fine_tuned[:,1], color="green", label="fine_tuned")
    plt.scatter(augmented[:,0], augmented[:,1], color="orange", label="augmented")
    plt.title(f"t-SNE of Last (11) Layer Embedding for the digit {i}")
    plt.legend()
    plt.tight_layout()
    plt.savefig(f'../Embedding_Graphs/{dataset_name}/{domain}/Digit_{i}', dpi=600)
    plt.show()

In [ ]:
tsne = TSNE(n_components=2, perplexity=5, random_state=42)

for i in range(num_classes):
    base = []
    fine_tuned = []
    augmented = []
    for j in range(20):
        base.append(base_embeddings[i][11][j])
        fine_tuned.append(fine_tuned_embeddings[i][11][j])
        augmented.append(augmented_embeddings[i][11][j])
    base = np.array(base)
    fine_tuned = np.array(fine_tuned)
    augmented = np.array(augmented)
    base = tsne.fit_transform(base)
    fine_tuned = tsne.fit_transform(fine_tuned)
    augmented = tsne.fit_transform(augmented)

    if i == num_classes-1:
        plt.scatter(base[:,0], base[:,1], color="blue", label="base")
        plt.scatter(fine_tuned[:,0], fine_tuned[:,1], color="green", label="fine_tuned")
        plt.scatter(augmented[:,0], augmented[:,1], color="orange", label="augmented")
    else:
        plt.scatter(base[:,0], base[:,1], color="blue")
        plt.scatter(fine_tuned[:,0], fine_tuned[:,1], color="green")
        plt.scatter(augmented[:,0], augmented[:,1], color="orange")

plt.title(f"t-SNE of Last (11) Layer Embedding for All Digits")
plt.legend()
plt.tight_layout()
plt.savefig(f'../Embedding_Graphs/{dataset_name}/{domain}/11th_Layer_All_Digits_20_Samples', dpi=600)
plt.show()